# Agent 04 - Refresh Ping Range Master (Polygon)

## Funcion

Construir/actualizar la tabla maestra `ping_range_master` desde Polygon para saber, por ticker:
- `has_data`
- `first_day`
- `last_day`
- `updated_at_utc`

## Como debemos hacerlo

1. Cargar universo de tickers (o delta) desde parquet local.
2. Hacer ping a `/v2/aggs/ticker/{ticker}/range/1/day/{start}/{end}` en `asc` y `desc` (`limit=1`).
3. Guardar resultado en parquet/csv maestra.
4. Ejecutar este notebook en modo **offline (1 vez al d?a)**, no en loop r?pido.


**Qué hace `agent_04_refresh_ping_range_master.ipynb`**

Ruta:

- C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\00_data_certification\agent_04_refresh_ping_range_master.ipynb

Función real:

1. Carga un universo de tickers (UNIVERSE_PARQUET).
2. Para cada ticker consulta Polygon en /v2/aggs/ticker/{ticker}/range/1/day/{PING_START}/{PING_END}:
    - sort=asc, limit=1 para first_day
    - sort=desc, limit=1 para last_day
3. Genera un master con:
    - ticker, has_data, first_day, last_day
4. Guarda:
    - ping_range_master.parquet
    - ping_range_master.csv

Es el agente que “pregunta a Polygon” el rango operativo real por ticker.


In [19]:
from pathlib import Path
import os
import time
import json
import requests
import pandas as pd

# --- Config ---
POLYGON_API_KEY = os.getenv("POLYGON_API_KEY", "")  # recomendado usar variable de entorno

# Universo fuente (ajusta al parquet real de tu universo)
UNIVERSE_PARQUET = Path(r"data\reference\universe_pti\hybrid_enriched\universe_hybrid_enriched_with_financial_ranges.parquet")

# Ventana para ping (operativa)
PING_START = "2004-01-01"
PING_END = "2026-12-31"

OUT_DIR = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference")
OUT_DIR.mkdir(parents=True, exist_ok=True)
PING_MASTER_PARQUET = OUT_DIR / "ping_range_master.parquet"
PING_MASTER_CSV = OUT_DIR / "ping_range_master.csv"

# Control de velocidad
SLEEP_PER_TICKER_SEC = 0.12
TIMEOUT_SEC = 30

print("UNIVERSE_PARQUET:", UNIVERSE_PARQUET)
print("PING_MASTER_PARQUET:", PING_MASTER_PARQUET)
print("POLYGON_API_KEY loaded:", bool(POLYGON_API_KEY))


UNIVERSE_PARQUET: data\reference\universe_pti\hybrid_enriched\universe_hybrid_enriched_with_financial_ranges.parquet
PING_MASTER_PARQUET: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.parquet
POLYGON_API_KEY loaded: True


In [18]:
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

fp = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\universe_pti\hybrid_enriched\universe_hybrid_enriched_with_financial_ranges.parquet")
df = pd.read_parquet(fp)

print(f"shape: {df.shape}")

print(df.head(1).T)

shape: (15979, 143)
                                                                       0
ticker                                                               AVE
name                                           AVENTIS, ADS(RP 1 ORD SH)
market                                                            stocks
locale                                                                us
primary_exchange                                                    XNYS
type                                                                  CS
active                                                              True
currency_name                                                        usd
cik                                                                  NaN
composite_figi                                                       NaN
share_class_figi                                                     NaN
list_date                                                           None
delisted_utc                   

In [20]:
# --- Cargar universo de tickers ---
if not UNIVERSE_PARQUET.exists():
    raise FileNotFoundError(f"No existe universo: {UNIVERSE_PARQUET}")

u = pd.read_parquet(UNIVERSE_PARQUET)

# detectar columna ticker
ticker_col = None
for c in ["ticker", "symbol", "Ticker", "SYMBOL"]:
    if c in u.columns:
        ticker_col = c
        break

if ticker_col is None:
    raise RuntimeError("No se encontr? columna ticker/symbol en el universo.")

tickers = (
    u[ticker_col]
    .astype(str)
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("ticker_col:", ticker_col)
print("tickers_total:", len(tickers))
print("sample:", tickers[:10])


ticker_col: ticker
tickers_total: 12468
sample: ['A', 'AA', 'AAA', 'AABA', 'AAC', 'AACB', 'AACI', 'AACQ', 'AACT', 'AAGR']


In [21]:
# --- Ping helpers ---
BASE = "https://api.polygon.io"


def _ping_side(ticker: str, sort: str):
    url = f"{BASE}/v2/aggs/ticker/{ticker}/range/1/day/{PING_START}/{PING_END}"
    params = {
        "adjusted": "true",
        "sort": sort,     # asc -> first_day, desc -> last_day
        "limit": 1,
        "apiKey": POLYGON_API_KEY,
    }
    r = requests.get(url, params=params, timeout=TIMEOUT_SEC)
    if r.status_code != 200:
        return {"ok": False, "status_code": r.status_code, "data": None, "error": r.text[:400]}

    js = r.json()
    results = js.get("results", [])
    if not results:
        return {"ok": True, "status_code": 200, "data": None, "error": None}

    t_ms = results[0].get("t")
    dt = pd.to_datetime(t_ms, unit="ms", utc=True).date() if t_ms is not None else None
    return {"ok": True, "status_code": 200, "data": str(dt) if dt is not None else None, "error": None}


def ping_ticker_range(ticker: str):
    asc = _ping_side(ticker, "asc")
    desc = _ping_side(ticker, "desc")

    has_data = bool(asc.get("data") and desc.get("data"))
    return {
        "ticker": ticker,
        "has_data": has_data,
        "first_day": asc.get("data"),
        "last_day": desc.get("data"),
        "asc_status": asc.get("status_code"),
        "desc_status": desc.get("status_code"),
        "asc_error": asc.get("error"),
        "desc_error": desc.get("error"),
    }


He creado:
```
- C:/TSIS_Data/02_backtest_SmallCaps/data_auditoria_polygon/cell_code/build_ping_range_master_resume.py
- C:/TSIS_Data/02_backtest_SmallCaps/data_auditoria_polygon/cell_code/run_build_ping_range_master_resume.ps1
```
Lanzadera con resume:
```sh
powershell -NoProfile -ExecutionPolicy Bypass -File C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\run_build_ping_range_master_resume.ps1 -Resume
```
Prueba corta primero:
```sh
powershell -NoProfile -ExecutionPolicy Bypass -File C:\TSIS_Data\v1\backtest_SmallCaps\notebooks\cell_code\run_build_ping_range_master_resume.ps1 -Resume -LimitTickers 100
```
Qué deja en disco mientras corre:
```
- C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.partial.parquet
- C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.partial.csv
- C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.progress.json
```
Al final:
```
- C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.parquet
- C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.csv
```
Cómo vigilar el progreso desde otra terminal:
```
Get-Content C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.progress.json -Raw
```
Y reanuda desde el parcial si existe.

In [22]:
# --- Ejecutar ping masivo ---
if not POLYGON_API_KEY:
    raise RuntimeError("Falta POLYGON_API_KEY. Exporta la variable de entorno antes de ejecutar.")

rows = []
for i, tk in enumerate(tickers, start=1):
    out = ping_ticker_range(tk)
    rows.append(out)

    if i % 100 == 0:
        print(f"processed={i}/{len(tickers)}")

    time.sleep(SLEEP_PER_TICKER_SEC)

ping_df = pd.DataFrame(rows)
ping_df["updated_at_utc"] = pd.Timestamp.utcnow().isoformat()

print("rows:", len(ping_df))
print("has_data true:", int(ping_df["has_data"].fillna(False).sum()))
print("has_data false:", int((~ping_df["has_data"].fillna(False)).sum()))

display(ping_df.head(10))


KeyboardInterrupt: 

In [23]:
from pathlib import Path
import json

p = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.progress.json")
print(json.dumps(json.loads(p.read_text(encoding="utf-8")), indent=2))

{
  "status": "completed",
  "processed": 100,
  "total": 100,
  "progress_pct": 100.0,
  "has_data_true": 97,
  "http_non_200": 0,
  "updated_at_utc": "2026-03-22T18:50:52.499716+00:00",
  "output_parquet": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\ping_range_master.parquet",
  "output_csv": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\ping_range_master.csv",
  "partial_parquet": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\ping_range_master.partial.parquet",
  "partial_csv": "C:\\TSIS_Data\\v1\\backtest_SmallCaps\\data\\reference\\ping_range_master.partial.csv"
}


In [24]:
import pandas as pd
from pathlib import Path

fp = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.parquet")
df = pd.read_parquet(fp)

print(f"path: {fp}")
print(f"shape: {df.shape}")
display(df.head(20))

path: C:\TSIS_Data\v1\backtest_SmallCaps\data\reference\ping_range_master.parquet
shape: (100, 9)


,ticker,has_data,first_day,last_day,asc_status,desc_status,asc_error,desc_error,updated_at_utc
0,A,True,2004-01-02,2026-03-20,200,200,None,None,2026-03-22T18:50:13.544977+00:00
1,AA,True,2004-01-02,2026-03-20,200,200,None,None,2026-03-22T18:50:13.935090+00:00
2,AAA,True,2004-01-02,2026-03-20,200,200,None,None,2026-03-22T18:50:14.316790+00:00
3,AABA,True,2017-06-19,2019-10-02,200,200,None,None,2026-03-22T18:50:14.703733+00:00
4,AAC,True,2004-01-02,2023-11-06,200,200,None,None,2026-03-22T18:50:15.083653+00:00
5,AACB,True,2004-01-02,2026-03-18,200,200,None,None,2026-03-22T18:50:15.470483+00:00
6,AACI,True,2021-11-10,2025-10-29,200,200,None,None,2026-03-22T18:50:15.846911+00:00
7,AACQ,True,2020-09-04,2021-06-24,200,200,None,None,2026-03-22T18:50:16.240454+00:00
8,AACT,True,2023-06-12,2025-09-24,200,200,None,None,2026-03-22T18:50:16.619085+00:00
9,AAGR,True,2023-12-07,2026-03-20,200,200,None,None,2026-03-22T18:50:17.004618+00:00


In [28]:
import pandas as pd
from pathlib import Path

fp = Path(r"C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet")
df = pd.read_parquet(fp)

print(f"path: {fp}")
print(f"shape: {df.shape}")
print(df.head(2).T)

path: C:\TSIS_Data\v1\backtest_SmallCaps\runs\backtest\market_cap_last_observed_cutoff\20260320_market_cap_last_observed_cutoff\market_cap_cutoff_lt_1b_active_inactive.parquet
shape: (4824, 16)
                                                        0                                1
ticker                                               AACT                             AAGR
first_seen_date                       2023-06-12 00:00:00              2023-12-07 00:00:00
last_observed_date                    2025-09-24 00:00:00              2024-09-25 00:00:00
status_rebuilt                                   inactive                         inactive
last_row_date                         2025-09-24 00:00:00              2024-09-25 00:00:00
anchor_date_used                      2025-09-24 00:00:00              2024-09-25 00:00:00
close_t                                              9.49                           0.1326
shares_outstanding_t                           55470890.5                     

In [ ]:
# --- Persistir master ---
ping_df.to_parquet(PING_MASTER_PARQUET, index=False)
ping_df.to_csv(PING_MASTER_CSV, index=False, encoding="utf-8")

print("saved:", PING_MASTER_PARQUET)
print("saved:", PING_MASTER_CSV)
